# Torghut strategy lifecycle

**TL;DR.** This notebook follows decisions into executions through the verified
`executions.trade_decision_id` key, reports linked and unlinked executions, and measures TCA coverage.
Rejected-signal outcomes remain a separate population because no stable signal-to-decision key exists.
It intentionally does not fabricate a signal/price/order overlay.

## Context and method

The default window is 30 days. Set `strategy_id` to a UUID to scope proven decision/execution lineage; rejection evidence remains unscoped and visibly separate.

In [ ]:
import json
import math

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, display
from plotly.subplots import make_subplots

from app.notebook_data import (
    Window,
    adapter_from_environment,
    capital_authority,
    execution_evidence,
    flow_snapshot,
    mode_banner,
    strategy_lifecycle,
)

adapter = globals().get('adapter') or adapter_from_environment()
banner = mode_banner(adapter)
banner_color = '#9a6700' if adapter.mode == 'fixture' else '#b42318'
display(HTML(f'''<div style="padding:14px 18px;border:2px solid {banner_color};border-radius:8px;
  font-weight:700;background:#fff8e6;margin-bottom:14px">{banner}</div>'''))

def bounded_points(frame, series_column, *, time_column='event_ts', preferred=5000):
    # Deterministically thin display points while preserving each series.
    if frame.empty or len(frame) <= preferred:
        return frame.copy()
    series_count = max(1, frame[series_column].nunique())
    per_series = max(2, preferred // series_count)
    chunks = []
    for _, group in frame.sort_values(time_column).groupby(series_column, sort=True):
        stride = max(1, math.ceil(len(group) / per_series))
        chunks.append(group.iloc[::stride])
    result = pd.concat(chunks, ignore_index=True)
    if len(result) > 10000:
        raise ValueError('figure point cap exceeded after deterministic thinning')
    return result

def bounded_rows(frame, *, preferred=5000):
    if frame.empty or len(frame) <= preferred:
        return frame.copy()
    stride = max(1, math.ceil(len(frame) / preferred))
    result = frame.iloc[::stride].head(10000).copy()
    if len(result) > 10000:
        raise ValueError('figure point cap exceeded after deterministic thinning')
    return result

def snapshot_frame(snapshot, name):
    if name not in snapshot.datasets:
        return pd.DataFrame()
    return snapshot.frame(name)

def empty_panel(message):
    display(HTML(f'<div style="padding:18px;border:1px solid #d0d5dd;border-radius:8px;color:#475467">{message}</div>'))

In [ ]:
strategy_id = None
snapshot = strategy_lifecycle(strategy_id, Window.days(30), adapter=adapter)
display(snapshot.provenance())
decisions = snapshot_frame(snapshot, 'decision_status')
executions = snapshot_frame(snapshot, 'execution_links')
rejections = snapshot_frame(snapshot, 'rejection_reasons')

## Decision status by strategy and day

In [ ]:
if decisions.empty:
    empty_panel('No decision rows: ' + '; '.join(snapshot.messages))
else:
    decisions['bucket_start'] = pd.to_datetime(decisions['bucket_start'], utc=True)
    decision_daily = decisions.groupby(['bucket_start', 'status'], as_index=False)['decision_count'].sum()
    fig = px.bar(decision_daily, x='bucket_start', y='decision_count', color='status',
                 title='Decision lifecycle statuses by day', barmode='stack',
                 labels={'bucket_start': 'UTC day', 'decision_count': 'decisions'})
    fig.update_layout(hovermode='x unified')
    fig.show()
    display(decisions.groupby(['strategy_id', 'status'], as_index=False)['decision_count'].sum()
            .sort_values('decision_count', ascending=False))

## Execution linkage, fills/cancels, and TCA coverage

In [ ]:
if executions.empty:
    empty_panel('No execution rows in the selected window.')
else:
    numeric = ['execution_count', 'linked_execution_count', 'unlinked_execution_count', 'tca_count']
    executions[numeric] = executions[numeric].apply(pd.to_numeric)
    linkage = executions[numeric].sum().to_frame('count').reset_index(names='measure')
    fig = px.bar(linkage, x='measure', y='count', color='measure',
                 title='Execution lineage and TCA coverage', text_auto=True)
    fig.update_layout(showlegend=False)
    fig.show()
    status_table = executions.groupby('status', as_index=False).agg(
        execution_count=('execution_count', 'sum'),
        linked_execution_count=('linked_execution_count', 'sum'),
        unlinked_execution_count=('unlinked_execution_count', 'sum'),
        tca_count=('tca_count', 'sum'),
        last_activity_at=('last_activity_at', 'max'),
    ).sort_values('execution_count', ascending=False)
    display(status_table)
    unlinked = executions[executions.unlinked_execution_count > 0]
    if not unlinked.empty:
        display(HTML('<div style="padding:12px;border:2px solid #b42318;border-radius:8px;font-weight:700">Unlinked executions require operator review.</div>'))
        display(unlinked)

## Rejected-signal outcomes (separate lineage)

In [ ]:
if rejections.empty:
    empty_panel('No rejected-signal outcome rows in the selected window.')
else:
    rejection_summary = rejections.groupby('reject_reason', as_index=False)['rejected_signal_count'].sum().nlargest(20, 'rejected_signal_count')
    fig = px.bar(rejection_summary, x='rejected_signal_count', y='reject_reason', orientation='h',
                 title='Rejected signals by reason — not joined to strategies')
    fig.update_layout(yaxis={'categoryorder': 'total ascending'})
    fig.show()
    display(rejections.sort_values('rejected_signal_count', ascending=False).head(50))

## Takeaways

Linked execution and TCA counts are evidence of lifecycle observability. Unlinked executions and rejection outcomes are reported explicitly; neither is silently attributed to a strategy.